# Exploratory uncertainty analysis on an open-weight language model

This notebook documents an experiment originally run in Google Colab using **Qwen/Qwen1.5-4B**, `ttomov/maqa_star` or `ttomov/ambigqa_star`, and Gaussian perturbations of a **top-10-token output-head subspace**.

**Important scientific limitation:** this is **not the QUAM algorithm**. It does not optimize adversarial models under a training-loss constraint, does not use QUAM's trajectory weights, and does not estimate an answer distribution. The conditional probabilities concern the next token **after a candidate answer is already in the prompt**.

The original run used `float16` and an epsilon of `1e-10` in logarithms. This may underflow, creating nonfinite uncertainty estimates. The default execution mode below retains the original arithmetic to facilitate tracing the historical experiment. An explicitly labeled alternate numerical mode is provided for **future reruns only**; it does not validate or replace results already reported in the paper.

To avoid accidentally launching a full-model computation, model inference is disabled until `RUN_INFERENCE=True`. Save the original `.pkl` data separately and **never overwrite those files** with new experimental runs.

## Configuration and Colab environment

In [ ]:
# Install dependencies in a fresh Colab environment if required:
# %pip install torch transformers accelerate datasets tqdm pandas seaborn matplotlib

from pathlib import Path
import pickle
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

MODEL_ID = "Qwen/Qwen1.5-4B"
DATASET = "ambigqa"  # "ambigqa" or "maqa"
DATASET_IDS = {"ambigqa": "ttomov/ambigqa_star", "maqa": "ttomov/maqa_star"}
SPLIT = "train"
TOP_K = 10
N_PERTURBATIONS = 30
NOISE_STD = 0.01
LOG_EPSILON = 1e-10

# Original computations used float16 for the model and math. Set to True only
# for a NEW experiment; its values must not be substituted for saved results.
STABLE_FLOAT32_MATH = False
RUN_INFERENCE = False
MAX_QUESTIONS = None  # For a quick smoke test, e.g. 5
OUTPUT_DIR = Path("/content/drive/MyDrive")
RESULT_PATH = OUTPUT_DIR / f"output_head_perturbation_{DATASET}_rerun.pkl"
print(f"Dataset: {DATASET_IDS[DATASET]} | mode: {'float32 stable' if STABLE_FLOAT32_MATH else 'historical float16'}")

### Google Drive (optional)

For Colab, mount Google Drive if you want to keep the rerun output beyond the runtime session. Otherwise, change `OUTPUT_DIR` to `Path("/content")` or another directory.

In [ ]:
# Run only in Google Colab when saving to MyDrive.
# from google.colab import drive
# drive.mount("/content/drive")

## Setting-B-style uncertainty calculation

The 4-D sampled tensor has shape `[batch, perturbations, 1, top_k]`. The reference prediction is a renormalized distribution on the selected 10 tokens. The original evaluation formulation is retained; the stable option changes only the *numerical implementation*, not the intended cross-entropy decomposition.

In [ ]:
def setting_b_components(reference_probs, sample_probs, *, stable=False, eps=1e-10):
    """Return total, reference entropy, and cross-entropy minus entropy.

    Sample shape: [N, S, 1, K]. A numerical rerun uses float32+clamp;
    historical mode leaves the original float16 arithmetic unchanged.
    """
    reference = reference_probs.float() if stable else reference_probs
    samples = sample_probs.float() if stable else sample_probs
    if stable:
        reference = reference.clamp_min(eps)
        reference = reference / reference.sum(dim=-1, keepdim=True)
        samples = samples.clamp_min(eps)
        samples = samples / samples.sum(dim=-1, keepdim=True)

    ref_expanded = reference.unsqueeze(1).unsqueeze(2)
    total = -(ref_expanded * torch.log(samples + eps)).sum(dim=-1).mean(dim=(1, 2))
    aleatoric = -((reference + eps) * torch.log(reference + eps)).sum(dim=-1)
    epistemic = total - aleatoric
    return total, aleatoric, epistemic


def sample_output_head_perturbations(topk_weights, hidden_state,
                                      *, count=30, noise_std=0.01):
    """Gaussian noise on a fixed top-K output-head submatrix (NOT QUAM)."""
    original = topk_weights.detach()
    draws = []
    with torch.no_grad():
        for _ in range(count):
            perturbed = original + torch.randn_like(original) * noise_std
            next_logits = hidden_state @ perturbed.T
            draws.append(F.softmax(next_logits, dim=-1).unsqueeze(1))
    return torch.stack(draws, dim=1)

# Cheap shape-only check; no model load or new results required.
ref_test = torch.tensor([[0.7, 0.3]], dtype=torch.float32)
samples_test = torch.tensor([[[[0.6, 0.4]], [[0.8, 0.2]]]], dtype=torch.float32)
assert all(x.shape == (1,) for x in setting_b_components(ref_test, samples_test))

## Optional generation of new outputs

This section is intentionally disabled by default. The run can be expensive, particularly across the entire AmbigQA* training split. In this pipeline, the candidate answer is embedded in the prompt, and a 30-member perturbation set is generated for the final hidden state. The saved output file uses a different name from the original historical pickle.

In [ ]:
def run_experiment():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from datasets import load_dataset

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto", torch_dtype=torch.float16
    )
    model.eval()
    dataset = load_dataset(DATASET_IDS[DATASET], split=SPLIT)
    n_questions = len(dataset) if MAX_QUESTIONS is None else min(MAX_QUESTIONS, len(dataset))

    results = []
    for index in tqdm(range(n_questions), desc=f"{DATASET} questions"):
        item = dataset[index]
        answers = [a[0] if isinstance(a, list) else a for a in item["answers"]]
        record = {"question": item["rephrased_question"], "variants": []}
        for answer in answers:
            prompt = f"Q: {record['question']}\nA: {answer}"
            encoded = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                hidden = model.model(**encoded).last_hidden_state[:, -1, :]
                logits = model.lm_head(hidden)
                full_probs = F.softmax(logits, dim=-1)
                top_probs, token_ids = torch.topk(full_probs, k=TOP_K, dim=-1)
                top_weights = model.lm_head.weight[token_ids[0]]
                ref_probs = F.softmax(hidden @ top_weights.T, dim=-1)
                samples = sample_output_head_perturbations(
                    top_weights, hidden, count=N_PERTURBATIONS, noise_std=NOISE_STD
                )
                total, al, ep = setting_b_components(
                    ref_probs, samples, stable=STABLE_FLOAT32_MATH, eps=LOG_EPSILON
                )
                record["variants"].append({
                    "answer": answer,
                    "total_uncertainty": total.item(),
                    "aleatoric_uncertainty": al.item(),
                    "epistemic_uncertainty": ep.item(),
                    "top_10_base_probs": top_probs[0].detach().cpu().numpy(),
                })
        results.append(record)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with RESULT_PATH.open("wb") as file:
        pickle.dump(results, file)
    print(f"Saved {len(results)} questions to {RESULT_PATH}")
    return results

if RUN_INFERENCE:
    generated_results = run_experiment()
else:
    print("Inference skipped. Set RUN_INFERENCE=True for a new model run.")

## Audit saved outputs before plotting

The historical files are required to reproduce the paper's coverage percentages. This cell can inspect a trusted `.pkl` stored on Google Drive. **Only unpickle files you generated yourself or obtained from a trusted collaborator.** All plotted correlations below use rows for which **every requested metric is finite**; filtering is explicit and the retained fraction is printed. This may not equal correlations in the original unfiltered notebook, especially for AmbigQA*.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Point to the historical file, for example:
# SAVED_RESULTS = Path("/content/drive/MyDrive/quam_ambigqa_results.pkl")
SAVED_RESULTS = RESULT_PATH  # Change to your actual trusted pickle for the paper's audit.


def flatten_results(path):
    with path.open("rb") as file:
        records = pickle.load(file)
    rows = []
    for q in records:
        for variant in q["variants"]:
            rows.append({
                "Epistemic": float(variant["epistemic_uncertainty"]),
                "Aleatoric": float(variant["aleatoric_uncertainty"]),
                "Total": float(variant["total_uncertainty"]),
                "Base_Probability": float(np.asarray(variant["top_10_base_probs"]).reshape(-1)[0]),
                "Top10_Mass": float(np.sum(np.asarray(variant["top_10_base_probs"]))),
            })
    return pd.DataFrame(rows), len(records)

if SAVED_RESULTS.exists():
    all_rows, question_count = flatten_results(SAVED_RESULTS)
    cols = ["Epistemic", "Aleatoric", "Total", "Base_Probability"]
    finite_mask = np.isfinite(all_rows[cols].to_numpy()).all(axis=1)
    valid = all_rows.loc[finite_mask, cols].copy()
    print(f"{question_count} questions | {len(all_rows)} variants | "
          f"{len(valid)} fully finite ({finite_mask.mean():.2%})")
    print(f"Median top-10 probability mass: {all_rows['Top10_Mass'].median():.3f}")
else:
    valid = pd.DataFrame()
    print(f"No file at {SAVED_RESULTS}; set SAVED_RESULTS to your existing trusted .pkl.")

## Descriptive correlations on the fully finite subset

In [ ]:
if not valid.empty:
    corr = valid.corr(method="pearson")
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", vmin=-1, vmax=1,
                cmap="coolwarm", linewidths=0.5, ax=ax)
    ax.set_title(f"{DATASET.upper()}: correlations on finite variants only")
    fig.tight_layout()
    plt.show()
    grid = sns.pairplot(valid, corner=True, diag_kind="hist",
                        plot_kws={"alpha": 0.35, "s": 12, "edgecolor": "none"})
    grid.figure.suptitle(f"{DATASET.upper()}: finite-only uncertainty metrics", y=1.02)
    plt.show()

## Interpretation and reproducibility notes

- **Historical output ≠ validated performance.** The paper reports 2,666/2,669 fully finite variants for MAQA* and 306/12,920 for AmbigQA* from the original saved files. This notebook's finite-only plots do not repair missing results.
- **Setting B** is a definition applied to these generated distributions. It does not imply that the sampling procedure is QUAM or that a QUAM importance weighting has been used.
- **Numerical mode:** `STABLE_FLOAT32_MATH=True` is a proposed numerical rerun. It must generate new saved data and be evaluated separately; never silently overwrite the original `.pkl` files.
- **Prompt semantics:** the prediction is the token *following* the supplied answer, not the probability of which answer is correct.
- **Next experimental requirements:** a well-defined answer-level target, suitable baseline, controls for truncation to top-10, actual adversarial QUAM search if claiming QUAM, and recorded seed/checkpoint/runtime configuration.
